#### Initialze

In [2]:
import sys
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
LEVEL2_BUCKET_PATH = PARENT / "server/out/places_level2"
LEVEL3_BUCKET_PATH = PARENT / "server/out/places_level3"
LEVEL3_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
LEVEL2_BUCKET = [f for f in LEVEL2_BUCKET_PATH.rglob("*.csv") if f.is_file()]
DF_LEVEL2 = pd.concat([pd.read_csv(f) for f in LEVEL2_BUCKET], ignore_index=True)

#### Parse JSON

In [3]:
import ast
from server.scripts.clean_places_level_3.wheelchair_level import wheelchair_level
df_level2_timetable = DF_LEVEL2[['id', 'regularOpeningHours']].copy()

df_level2_timetable['openningHours'] = df_level2_timetable['regularOpeningHours'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else None)
df_level2_timetable.drop(columns=["regularOpeningHours"], inplace=True)
df_level2_timetable['openningHours']= df_level2_timetable['openningHours'].apply(lambda x: x['periods'] if isinstance(x, dict) and 'periods' in x.keys() else None)
df_level2_timetable.dropna(subset=['openningHours'], inplace=True)
df_level2_timetable.to_csv(LEVEL3_BUCKET_PATH / 'timetable.csv', index=False)